In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import itertools
import wandb

# utils

In [ ]:
def get_color_cycle(algorithms):
  """
  Creates a circular buffer of colors for plotting multiple algorithms.

  Args:
    algorithms: A list of algorithm names.

  Returns:
    A dictionary mapping each algorithm to a unique color.
  """
  colors = itertools.cycle(plt.colormaps.get_cmap('tab10').colors)  # You can choose a different colormap
  color_map = {}
  for algorithm in algorithms:
    color_map[algorithm] = next(colors)
  return color_map


# Create a custom handler for vertical lines in legend
class VerticalLineHandler:
    def legend_artist(self, legend, orig_handle, fontsize, handlebox):
        x0, y0 = handlebox.xdescent, handlebox.ydescent
        width = handlebox.width
        height = handlebox.height
        # Create a vertical line in the center of the box
        line = plt.Line2D([x0 + width/2, x0 + width/2],
                         [y0, y0 + height],
                         color=orig_handle.get_color(),
                         linestyle=orig_handle.get_linestyle())
        handlebox.add_artist(line)
        return line


# formatting stuff:
title_font_dict = {'weight': 'bold', 'size': 30}
axis_font_dict = {'weight': 'bold', 'size': 18}
legend_font_dict = {'weight': 'bold', 'size': 14}


# setup

In [ ]:
api = wandb.Api()

In [ ]:
runs = api.runs("jfcevallos/TIGER2.0")

In [ ]:
default_run_id = "s6vvvyir"

for run in runs:
    if run.id == default_run_id:
        print(run.name, run.id)
        default_run_name = run.name
        default_run = run
        # print('scanning run history:')
        # default_run_history = list(run.scan_history(keys=None))
        # print('creating dataframe:')
        # default_run_history = pd.DataFrame(history_rows)
        default_run_history = run.history(samples=20000)
        default_run_history.set_index("_runtime", inplace=True)
        break

# ASAP Architectures

In [ ]:
asap_runs = {}
for run in runs:
    if run.group == "arch_agency":
        print(run.name, run.id)
        asap_runs[run.name] = {
            "id": run.id}
        # print('scanning run history:')
        # history_rows = list(run.scan_history(keys=None))
        # print('creating dataframe:')
        # asap_runs[run.name]["history"] = pd.DataFrame(history_rows)
        asap_runs[run.name]["history"] = run.history(samples=20000)
        asap_runs[run.name]["history"].set_index("_runtime", inplace=True)


del asap_runs['agency_optim']
asap_runs["agency_simple_krloss"]["display_name"] = "Distance_KR + BCELoss"
asap_runs["agency_dotprod_kr"]["display_name"] = "DotProd_KR + UMAPLoss"
asap_runs["agency_dotprod_kr_simplekrloss"]["display_name"] = "DotProd_KR + BCELoss"


default_run_id = "eu5i235u"

for run in runs:
    if run.id == default_run_id:
        print(run.name, run.id)
        default_run_name = run.name
        default_run = run
        # print('scanning run history:')
        # default_run_history = list(run.scan_history(keys=None))
        # print('creating dataframe:')
        # default_run_history = pd.DataFrame(history_rows)
        default_run_history = run.history(samples=20000)
        default_run_history.set_index("_runtime", inplace=True)
        break


asap_runs[default_run_name] = {
    "id": default_run_id,
    "history": default_run_history,
    "display_name": "Distance_KR + UMAPLoss (default)"
}


## backlog templates

In [ ]:

# [col for col in list(default_run_history.columns) if "AGENT" in col]

plt.figure(figsize=(20, 5))
plt.subplot(1, 1, 1)

# Adjustable smoothing parameters
ROLLING_WINDOW_SIZE = 150 # Number of samples for rolling calculations

algorithms = [run_dict["display_name"] if "display_name" in run_dict else run_name for run_name, run_dict in asap_runs.items()]
color_map = get_color_cycle(algorithms)

for run_name, run_dict in asap_runs.items():
    display_name = run_dict.get("display_name", run_name)
    history_df = run_dict["history"].copy() # Work on a copy to avoid modifying original data

    if "AGENT/generic_reward" in history_df.columns and not history_df["AGENT/generic_reward"].empty:
        reward_series = history_df["AGENT/generic_reward"]

        # Calculate running average (instead of EMA)
        running_avg_series = reward_series.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()

        # Calculate rolling 1st and 3rd quantiles from the original reward series
        q1_series = reward_series.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.25)
        q3_series = reward_series.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.75)

        # Plot Running Average line with specified alpha
        plt.plot(running_avg_series.index, running_avg_series,
                 # label=f'{display_name} (Running Avg, window={ROLLING_WINDOW_SIZE})',
                 label=f'{display_name}',
                 color=color_map[display_name],
                 linewidth=2,
                 alpha=0.8)

        # Plot 1st and 3rd quantiles
        plt.plot(q1_series.index, q1_series,
                 #label=f'{display_name} (1st Q)',
                 color=color_map[display_name],
                 linestyle='--',
                 alpha=0.5,
                 linewidth=0.5)
        plt.plot(q3_series.index, q3_series,
                 #label=f'{display_name} (3rd Q)',
                 color=color_map[display_name],
                 linestyle='--',
                 alpha=0.5,
                 linewidth=0.5)

# Make axis numbers big and bold
plt.tick_params(axis='x', labelsize=axis_font_dict['size'])
plt.tick_params(axis='y', labelsize=axis_font_dict['size'])

# Make tick labels bold
for tick in plt.gca().get_xticklabels():
    tick.set_fontweight('bold')
for tick in plt.gca().get_yticklabels():
    tick.set_fontweight('bold')

plt.title('Rewards for different ASAP Architectures', **title_font_dict)
plt.xlabel('Training Time (seconds)', **axis_font_dict)
plt.ylabel('Rewards', **axis_font_dict)
plt.ylim(-1.5, 2) # Set y-axis limits
plt.legend(prop=legend_font_dict)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
metrics_to_plot = [
    {"key": "AGENT/generic_reward", "title": "Rewards", "ylim": (-1.5, 2), "rolling_window_size": 150},
    {"key": "mean_episode_budget", "title": "Mean Episode Budget", "ylim": (-50, 50), "rolling_window_size": 500}, # Adjusted ylim
    {"key": "epistemic_actions_per_episode", "title": "Epistemic Actions per Episode", "ylim": (0, 12), "rolling_window_size": 500}
]

max_secs_to_plot = 45 * 60

algorithms = [run_dict["display_name"] if "display_name" in run_dict else run_name for run_name, run_dict in asap_runs.items()]
color_map = get_color_cycle(algorithms)

fig, axes = plt.subplots(len(metrics_to_plot), 1, figsize=(20, 5 * len(metrics_to_plot)), sharex=False)

for i, metric_info in enumerate(metrics_to_plot):
    metric_key = metric_info["key"]
    metric_title = metric_info["title"]
    metric_ylim = metric_info["ylim"]
    ROLLING_WINDOW_SIZE = metric_info["rolling_window_size"]
    ax = axes[i]

    for run_name, run_dict in asap_runs.items():
        display_name = run_dict.get("display_name", run_name)
        history_df = run_dict["history"].copy()

        # Trim the timeseries data up to max_secs_to_plot seconds
        history_df = history_df[history_df.index <= max_secs_to_plot]

        if metric_key in history_df.columns and not history_df[metric_key].empty:
            data_series = history_df[metric_key]

            # Ensure data is numeric before interpolation
            data_series_numeric = pd.to_numeric(data_series, errors='coerce')

            # Interpolate discontinuities and address FutureWarning
            data_series_interpolated = data_series_numeric.infer_objects(copy=False).interpolate(method='linear')

            # Calculate running average
            running_avg_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()

            # Calculate rolling 1st and 3rd quantiles from the interpolated data series
            q1_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.25)
            q3_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.75)

            # Plot Running Average line
            ax.plot(running_avg_series.index, running_avg_series,
                     label=f'{display_name}',
                     color=color_map[display_name],
                     linewidth=2,
                     alpha=0.8)

            # Plot 1st and 3rd quantiles
            ax.plot(q1_series.index, q1_series,
                     color=color_map[display_name],
                     linestyle='--',
                     alpha=0.5,
                     linewidth=0.5)
            ax.plot(q3_series.index, q3_series,
                     color=color_map[display_name],
                     linestyle='--',
                     alpha=0.5,
                     linewidth=0.5)

    # Add horizontal line for 'epistemic_actions_per_episode' if it's the current metric
    if metric_key == "epistemic_actions_per_episode":
        ax.axhline(y=7, color='red', linestyle='--', label='optim')

    # Make axis numbers big and bold
    ax.tick_params(axis='x', labelsize=axis_font_dict['size'])
    ax.tick_params(axis='y', labelsize=axis_font_dict['size'])

    # Make tick labels bold
    for tick in ax.get_xticklabels():
        tick.set_fontweight('bold')
    for tick in ax.get_yticklabels():
        tick.set_fontweight('bold')

    ax.set_title(f'{metric_title} for different ASAP Architectures', **title_font_dict)
    ax.set_xlabel('Training Time (seconds)', **axis_font_dict)
    ax.set_ylabel(metric_title, **axis_font_dict)
    if metric_ylim:
        ax.set_ylim(metric_ylim) # Set y-axis limits if provided
    ax.legend(prop=legend_font_dict)
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
metrics_to_plot = [
    {"key": "Inference/KR_NMI", "title": "Kernel Regression NMI", "ylim": (0, 1.05), "rolling_window_size": 350},
    {"key": "Inference/AD Acc", "title": "Anomaly Detection Accuracy", "ylim": (0.5, 0.8), "rolling_window_size": 20}, # Adjusted ylim
    {"key": "Inference/Acc", "title": "Classification Accuracy", "ylim": (0, 1.05), "rolling_window_size": 450}
]

algorithms = [run_dict["display_name"] if "display_name" in run_dict else run_name for run_name, run_dict in asap_runs.items()]
color_map = get_color_cycle(algorithms)

fig, axes = plt.subplots(len(metrics_to_plot), 1, figsize=(20, 5 * len(metrics_to_plot)), sharex=False)

for i, metric_info in enumerate(metrics_to_plot):
    metric_key = metric_info["key"]
    metric_title = metric_info["title"]
    metric_ylim = metric_info["ylim"]
    ROLLING_WINDOW_SIZE = metric_info["rolling_window_size"]
    ax = axes[i]

    for run_name, run_dict in asap_runs.items():
        display_name = run_dict.get("display_name", run_name)
        history_df = run_dict["history"].copy()

        # Trim the timeseries data up to max_secs_to_plot seconds
        history_df = history_df[history_df.index <= max_secs_to_plot]

        if metric_key in history_df.columns and not history_df[metric_key].empty:
            data_series = history_df[metric_key]

            # Ensure data is numeric before interpolation
            data_series_numeric = pd.to_numeric(data_series, errors='coerce')

            # Interpolate discontinuities and address FutureWarning
            data_series_interpolated = data_series_numeric.infer_objects(copy=False).interpolate(method='linear')

            # Calculate running average
            running_avg_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()

            # Calculate rolling 1st and 3rd quantiles from the interpolated data series
            q1_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.25)
            q3_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.75)

            # Plot Running Average line
            ax.plot(running_avg_series.index, running_avg_series,
                     label=f'{display_name}',
                     color=color_map[display_name],
                     linewidth=2,
                     alpha=0.8)

            # Plot 1st and 3rd quantiles
            ax.plot(q1_series.index, q1_series,
                     color=color_map[display_name],
                     linestyle='--',
                     alpha=0.5,
                     linewidth=0.5)
            ax.plot(q3_series.index, q3_series,
                     color=color_map[display_name],
                     linestyle='--',
                     alpha=0.5,
                     linewidth=0.5)


    # Make axis numbers big and bold
    ax.tick_params(axis='x', labelsize=axis_font_dict['size'])
    ax.tick_params(axis='y', labelsize=axis_font_dict['size'])

    # Make tick labels bold
    for tick in ax.get_xticklabels():
        tick.set_fontweight('bold')
    for tick in ax.get_yticklabels():
        tick.set_fontweight('bold')

    ax.set_title(f'{metric_title} for different ASAP Architectures', **title_font_dict)
    ax.set_xlabel('Training Time (seconds)', **axis_font_dict)
    ax.set_ylabel(metric_title, **axis_font_dict)
    if metric_ylim:
        ax.set_ylim(metric_ylim) # Set y-axis limits if provided
    ax.legend(prop=legend_font_dict)
    ax.grid(True)

plt.tight_layout()
plt.show()

## final plot:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import itertools
import wandb
from matplotlib.lines import Line2D # Import Line2D for custom legend handles

# Metrics from the first plot (Rewards, Budget, Epistemic Actions)
metrics_to_plot_part1 = [
    {"key": "AGENT/generic_reward", "title": "Rewards", "ylim": (-0.5, 0.75), "rolling_window_size": 400},
    {"key": "mean_episode_budget", "title": "Mean Episode Budget", "ylim": (-20, 30), "rolling_window_size": 400},
    {"key": "active_inference/value_loss", "title": "Critic Loss", "ylim": (0, 0.6), "rolling_window_size": 100}
]

# Metrics from the second plot (Inference metrics)
metrics_to_plot_part2 = [
    {"key": "Inference/KR_NMI", "title": "Kernel Regression NMI", "ylim": (0, 1.05), "rolling_window_size": 400},
    {"key": "Inference/AD Acc", "title": "Anomaly Detection Accuracy", "ylim": (0.5, 0.8), "rolling_window_size": 20},
    {"key": "Inference/Acc", "title": "Classification Accuracy", "ylim": (0, 1.05), "rolling_window_size": 400}
]

# Combine all metrics for the 3x2 grid
metrics_to_plot = metrics_to_plot_part1 + metrics_to_plot_part2

# max_secs_to_plot from the original code for the first set of plots
max_secs_to_plot = 2 * 60 * 60 # 7200 seconds

algorithms = [run_dict["display_name"] if "display_name" in run_dict else run_name for run_name, run_dict in asap_runs.items()]
color_map = get_color_cycle(algorithms)

# Create custom legend handles for the entire figure, based on the algorithms
all_legend_handles = []
all_legend_labels = []
for algo in algorithms:
    all_legend_handles.append(Line2D([0], [0], color=color_map[algo], lw=2, alpha=0.8))
    all_legend_labels.append(algo)

# Add the 'optim' horizontal line to the legend if it's relevant for 'epistemic_actions_per_episode'
if any(m['key'] == 'epistemic_actions_per_episode' for m in metrics_to_plot):
    all_legend_handles.append(Line2D([0], [0], color='red', linestyle='--', lw=1, alpha=0.5))
    all_legend_labels.append('optim no. of epistemic actions')


# Create the figure and a 3x2 grid of subplots
fig, axes = plt.subplots(3, 2, figsize=(20, 12), sharex=False) # Adjust figsize for better readability
axes_flat = axes.flatten() # Flatten the 2D array of axes for easier iteration

for i, metric_info in enumerate(metrics_to_plot):
    metric_key = metric_info["key"]
    metric_title = metric_info["title"]
    metric_ylim = metric_info["ylim"]
    ROLLING_WINDOW_SIZE = metric_info["rolling_window_size"]
    ax = axes_flat[i] # Get the current subplot axis

    for run_name, run_dict in asap_runs.items():
        display_name = run_dict.get("display_name", run_name)
        history_df = run_dict["history"].copy()

        # Trim the timeseries data up to max_secs_to_plot seconds
        history_df = history_df[history_df.index <= max_secs_to_plot]

        if metric_key in history_df.columns and not history_df[metric_key].empty:
            data_series = history_df[metric_key]

            # Ensure data is numeric before interpolation
            data_series_numeric = pd.to_numeric(data_series, errors='coerce')

            # Interpolate discontinuities and address FutureWarning
            data_series_interpolated = data_series_numeric.infer_objects(copy=False).interpolate(method='linear')

            # Calculate running average
            running_avg_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()

            # Calculate rolling 1st and 3rd quantiles from the interpolated data series
            q1_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.25)
            q3_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.75)

            # Plot Running Average line
            ax.plot(running_avg_series.index, running_avg_series,
                     color=color_map[display_name],
                     linewidth=2,
                     alpha=0.8)

            # Plot 1st and 3rd quantiles
            # ax.plot(q1_series.index, q1_series,
            #          color=color_map[display_name],
            #          linestyle='--',
            #          alpha=0.5,
            #          linewidth=0.5)
            # ax.plot(q3_series.index, q3_series,
            #          color=color_map[display_name],
            #          linestyle='--',
            #          alpha=0.5,
            #          linewidth=0.5)

    # Add horizontal line for 'epistemic_actions_per_episode' if it's the current metric
    if metric_key == "epistemic_actions_per_episode":
        ax.axhline(y=7, color='red', linestyle='--', label='optim')

    # Make axis numbers big and bold
    ax.tick_params(axis='y', labelsize=axis_font_dict['size'])

    # Make tick labels bold
    for tick in ax.get_yticklabels():
        tick.set_fontweight('bold')

    # Control x-axis labels: only for the bottom row
    if i >= (len(metrics_to_plot) - 2): # Check if it's one of the bottom two plots
        ax.tick_params(axis='x', labelsize=axis_font_dict['size'], labelbottom=True) # Ensure labels are shown
        for tick in ax.get_xticklabels():
            tick.set_fontweight('bold')
        ax.set_xlabel('Training Time (seconds)', **axis_font_dict)
    else:
        ax.tick_params(axis='x', labelbottom=False) # Hide x-axis labels for other subplots

    ax.set_title(metric_title, **title_font_dict)
    # ax.set_ylabel(metric_title, **axis_font_dict)
    if metric_ylim:
        ax.set_ylim(metric_ylim) # Set y-axis limits if provided
    ax.grid(True)

# Add a single legend to the right of the entire plot
fig.legend(all_legend_handles, all_legend_labels,
           prop=legend_font_dict,
           loc='center right', # Position the legend outside the plots
           bbox_to_anchor=(0.98, 0.12), # Example coordinates to place it outside
           title='Algorithms',
           title_fontsize=legend_font_dict['size'] + 2,
           ncol=2)

# Add a super title for the entire figure
fig.suptitle('Performance Metrics for ASAP Architectures', **title_font_dict, y=1.02)

# Adjust layout to make space for the suptitle and legend
plt.tight_layout(rect=[0, 0, 1, 1.03]) # Adjust right and top boundary
plt.show()

# Ablating Neural Modules

In [ ]:
ablating_modules_runs = {}
for run in runs:
    if run.group == "ablating_modules":
        print(run.name, run.id)
        ablating_modules_runs[run.name] = {
            "id": run.id}
        # print('scanning run history:')
        # history_rows = list(run.scan_history(keys=None))
        # print('creating dataframe:')
        # ablating_modules_runs[run.name]["history"] = pd.DataFrame(history_rows)
        ablating_modules_runs[run.name]["history"] = run.history(samples=20000)
        ablating_modules_runs[run.name]["history"].set_index("_runtime", inplace=True)


ablating_modules_runs["perfect_inference"]["display_name"] = "Perfect IM"
ablating_modules_runs["only_neural_AD"]["display_name"] = "Only Neural Anomaly Detection"
ablating_modules_runs["only_neural_CS"]["display_name"] = "Only Neural Classification"
ablating_modules_runs["only_neural_KR"]["display_name"] = "Only Neural Clustering"
ablating_modules_runs["not_neural_CS"]["display_name"] = "Not neural Classification"
ablating_modules_runs["not_neural_KR"]["display_name"] = "Not neural Clustering"
ablating_modules_runs["not_neural_AD"]["display_name"] = "Not neural Anomaly Detection"

ablating_modules_runs[default_run_name] = {
    "id": default_run_id,
    "history": default_run_history,
    "display_name": "Neural outputs (Default)"
}

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import itertools
import wandb
from matplotlib.lines import Line2D # Import Line2D for custom legend handles

# Metrics from the first plot (Rewards, Budget, Epistemic Actions)
metrics_to_plot_part1 = [
    {"key": "AGENT/generic_reward", "title": "Rewards", "ylim": (-1.5, 2), "rolling_window_size": 400},
    {"key": "mean_episode_budget", "title": "Mean Episode Budget", "ylim": (-20, 25), "rolling_window_size": 400},
    {"key": "epistemic_actions_per_episode", "title": "Epistemic Actions per Episode", "ylim": (0, 12), "rolling_window_size": 400}
]

# Metrics from the second plot (Inference metrics)
metrics_to_plot_part2 = [
    {"key": "Inference/KR_NMI", "title": "Kernel Regression NMI", "ylim": (0, 1.05), "rolling_window_size": 400},
    {"key": "Inference/AD Acc", "title": "Anomaly Detection Accuracy", "ylim": (0.5, 1.05), "rolling_window_size": 20},
    {"key": "Inference/Acc", "title": "Classification Accuracy", "ylim": (0, 1.05), "rolling_window_size": 400}
]

# Combine all metrics for the legend check and general overview (not for direct iteration over axes)
metrics_to_plot = metrics_to_plot_part1 + metrics_to_plot_part2

# max_secs_to_plot from the original code for the first set of plots
max_secs_to_plot = 60 * 60 # 2700 seconds

algorithms = [run_dict["display_name"] if "display_name" in run_dict else run_name for run_name, run_dict in ablating_modules_runs.items()]
color_map = get_color_cycle(algorithms)

# Create custom legend handles for the entire figure, based on the algorithms
all_legend_handles = []
all_legend_labels = []
for algo in algorithms:
    all_legend_handles.append(Line2D([0], [0], color=color_map[algo], lw=2, alpha=0.8))
    all_legend_labels.append(algo)

# Add the 'optim' horizontal line to the legend if it's relevant for 'epistemic_actions_per_episode'
if any(m['key'] == 'epistemic_actions_per_episode' for m in metrics_to_plot):
    all_legend_handles.append(Line2D([0], [0], color='red', linestyle='--', lw=1, alpha=0.5))
    all_legend_labels.append('optim no. of epistemic actions')


# Create the figure and a 3x2 grid of subplots
fig, axes = plt.subplots(3, 2, figsize=(20, 12), sharex=False) # Adjust figsize for better readability

# Loop through each column (part1 for col 0, part2 for col 1)
for col_idx, current_metrics_list in enumerate([metrics_to_plot_part1, metrics_to_plot_part2]):
    # Loop through each metric in the current column's list
    for row_idx, metric_info in enumerate(current_metrics_list):
        metric_key = metric_info["key"]
        metric_title = metric_info["title"]
        metric_ylim = metric_info["ylim"]
        ROLLING_WINDOW_SIZE = metric_info["rolling_window_size"]
        ax = axes[row_idx, col_idx] # Get the current subplot axis based on row and column

        for run_name, run_dict in ablating_modules_runs.items():
            display_name = run_dict.get("display_name", run_name)
            history_df = run_dict["history"].copy()

            # Trim the timeseries data up to max_secs_to_plot seconds
            history_df = history_df[history_df.index <= max_secs_to_plot]

            if metric_key in history_df.columns and not history_df[metric_key].empty:
                data_series = history_df[metric_key]

                # Ensure data is numeric before interpolation
                data_series_numeric = pd.to_numeric(data_series, errors='coerce')

                # Interpolate discontinuities and address FutureWarning
                data_series_interpolated = data_series_numeric.infer_objects(copy=False).interpolate(method='linear')

                # Calculate running average
                running_avg_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()

                # Calculate rolling 1st and 3rd quantiles from the interpolated data series
                q1_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.25)
                q3_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.75)

                # Plot Running Average line
                ax.plot(running_avg_series.index, running_avg_series,
                         color=color_map[display_name],
                         linewidth=2,
                         alpha=0.8)

                # Plot 1st and 3rd quantiles
                ax.plot(q1_series.index, q1_series,
                         color=color_map[display_name],
                         linestyle='--',
                         alpha=0.5,
                         linewidth=0.5)
                ax.plot(q3_series.index, q3_series,
                         color=color_map[display_name],
                         linestyle='--',
                         alpha=0.5,
                         linewidth=0.5)

        # Add horizontal line for 'epistemic_actions_per_episode' if it's the current metric
        if metric_key == "epistemic_actions_per_episode":
            ax.axhline(y=7, color='red', linestyle='--', label='optim')

        # Make axis numbers big and bold
        ax.tick_params(axis='y', labelsize=axis_font_dict['size'])

        # Make tick labels bold
        for tick in ax.get_yticklabels():
            tick.set_fontweight('bold')

        # Control x-axis labels: only for the bottom row (row_idx == 2)
        if row_idx == 2: # Check if it's the bottom row
            ax.tick_params(axis='x', labelsize=axis_font_dict['size'], labelbottom=True) # Ensure labels are shown
            for tick in ax.get_xticklabels():
                tick.set_fontweight('bold')
            ax.set_xlabel('Training Time (seconds)', **axis_font_dict)
        else:
            ax.tick_params(axis='x', labelbottom=False) # Hide x-axis labels for other subplots

        ax.set_title(metric_title, **title_font_dict)
        # ax.set_ylabel(metric_title, **axis_font_dict)
        if metric_ylim:
            ax.set_ylim(metric_ylim) # Set y-axis limits if provided
        ax.grid(True)

# Add a single legend to the right of the entire plot
fig.legend(all_legend_handles, all_legend_labels,
           prop=legend_font_dict,
           loc='center right', # Position the legend outside the plots
           bbox_to_anchor=(0.85, -0.05), # Example coordinates to place it outside
           title='Legend',
           title_fontsize=legend_font_dict['size'] + 2,
           ncol=3)

# Add a super title for the entire figure
fig.suptitle('Neural Model Ablation', **title_font_dict, y=1.02)

# Adjust layout to make space for the suptitle and legend
plt.tight_layout(rect=[0, 0, 1, 1.03]) # Adjust right and top boundary
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import itertools
import wandb
from matplotlib.lines import Line2D # Import Line2D for custom legend handles

# Metrics from the first plot (Rewards, Budget, Epistemic Actions)
metrics_to_plot_part1 = [
    {"key": "ml_profiling_millis/train_inf_module_single_batch", "title": "Single Batch IM Training Time (ms) ", "ylim": (260, 850), "rolling_window_size": 400},
    {"key": "ml_profiling_millis/process_input_total", "title": "Total Processing per Input Batch (ms) ", "ylim": (500, 1250), "rolling_window_size": 400},
]

# Metrics from the second plot (Inference metrics)
metrics_to_plot_part2 = [
    {"key": "ml_profiling_millis/online_inference_total", "title": "Inference Time (ms)", "ylim": (310, 380), "rolling_window_size": 400},
    {"key": "controller_metrics/raw_cpu_percent", "title": "CPU Percent (1CPU-100%)", "ylim": (680, 710), "rolling_window_size": 400},
]

# Combine all metrics for the legend check and general overview (not for direct iteration over axes)
metrics_to_plot = metrics_to_plot_part1 + metrics_to_plot_part2

# max_secs_to_plot from the original code for the first set of plots
max_secs_to_plot = 60 * 60 # 2700 seconds

algorithms = [run_dict["display_name"] if "display_name" in run_dict else run_name for run_name, run_dict in ablating_modules_runs.items()]
color_map = get_color_cycle(algorithms)

# Create custom legend handles for the entire figure, based on the algorithms
all_legend_handles = []
all_legend_labels = []
for algo in algorithms:
    all_legend_handles.append(Line2D([0], [0], color=color_map[algo], lw=2, alpha=0.8))
    all_legend_labels.append(algo)

# Add the 'optim' horizontal line to the legend if it's relevant for 'epistemic_actions_per_episode'
if any(m['key'] == 'epistemic_actions_per_episode' for m in metrics_to_plot):
    all_legend_handles.append(Line2D([0], [0], color='red', linestyle='--', lw=1, alpha=0.5))
    all_legend_labels.append('optim no. of epistemic actions')


# Create the figure and a 3x2 grid of subplots
fig, axes = plt.subplots(len(metrics_to_plot_part1), 2, figsize=(20, 8), sharex=False) # Adjust figsize for better readability

# Loop through each column (part1 for col 0, part2 for col 1)
for col_idx, current_metrics_list in enumerate([metrics_to_plot_part1, metrics_to_plot_part2]):
    # Loop through each metric in the current column's list
    for row_idx, metric_info in enumerate(current_metrics_list):
        metric_key = metric_info["key"]
        metric_title = metric_info["title"]
        metric_ylim = metric_info["ylim"]
        ROLLING_WINDOW_SIZE = metric_info["rolling_window_size"]
        ax = axes[row_idx, col_idx] # Get the current subplot axis based on row and column

        for run_name, run_dict in ablating_modules_runs.items():
            display_name = run_dict.get("display_name", run_name)
            history_df = run_dict["history"].copy()

            # Trim the timeseries data up to max_secs_to_plot seconds
            history_df = history_df[history_df.index <= max_secs_to_plot]

            if metric_key in history_df.columns and not history_df[metric_key].empty:
                data_series = history_df[metric_key]

                # Ensure data is numeric before interpolation
                data_series_numeric = pd.to_numeric(data_series, errors='coerce')

                # Interpolate discontinuities and address FutureWarning
                data_series_interpolated = data_series_numeric.infer_objects(copy=False).interpolate(method='linear')

                # Calculate running average
                running_avg_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()

                # Calculate rolling 1st and 3rd quantiles from the interpolated data series
                q1_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.25)
                q3_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.75)

                # Plot Running Average line
                ax.plot(running_avg_series.index, running_avg_series,
                         color=color_map[display_name],
                         linewidth=2,
                         alpha=0.8)

                # Plot 1st and 3rd quantiles
                # ax.plot(q1_series.index, q1_series,
                #          color=color_map[display_name],
                #          linestyle='--',
                #          alpha=0.5,
                #          linewidth=0.5)
                # ax.plot(q3_series.index, q3_series,
                #          color=color_map[display_name],
                #          linestyle='--',
                #          alpha=0.5,
                #          linewidth=0.5)

        # Add horizontal line for 'epistemic_actions_per_episode' if it's the current metric
        if metric_key == "epistemic_actions_per_episode":
            ax.axhline(y=7, color='red', linestyle='--', label='optim')

        # Make axis numbers big and bold
        ax.tick_params(axis='y', labelsize=axis_font_dict['size'])

        # Make tick labels bold
        for tick in ax.get_yticklabels():
            tick.set_fontweight('bold')

        # Control x-axis labels: only for the bottom row (row_idx == 2)
        if row_idx == 1: # Check if it's the bottom row
            ax.tick_params(axis='x', labelsize=axis_font_dict['size'], labelbottom=True) # Ensure labels are shown
            for tick in ax.get_xticklabels():
                tick.set_fontweight('bold')
            ax.set_xlabel('Training Time (seconds)', **axis_font_dict)
        else:
            ax.tick_params(axis='x', labelbottom=False) # Hide x-axis labels for other subplots

        ax.set_title(metric_title, **title_font_dict)
        # ax.set_ylabel(metric_title, **axis_font_dict)
        if metric_ylim:
            ax.set_ylim(metric_ylim) # Set y-axis limits if provided
        ax.grid(True)

# Add a single legend to the right of the entire plot
fig.legend(all_legend_handles, all_legend_labels,
           prop=legend_font_dict,
           loc='center right', # Position the legend outside the plots
           bbox_to_anchor=(0.85, -0.08), # Example coordinates to place it outside
           title='Neural model / Oracle configuration',
           title_fontsize=legend_font_dict['size'] + 2,
           ncol=3)

# Add a super title for the entire figure
fig.suptitle('System Runtime Metrics', **title_font_dict, y=1.02)

# Adjust layout to make space for the suptitle and legend
plt.tight_layout(rect=[0, 0, 1, 1.03]) # Adjust right and top boundary
plt.show()

# Confidence Strategy

In [ ]:
conf_runs = {}
for run in runs:
    if run.group == "confidence_strategies":
        print(run.name, run.id)
        conf_runs[run.name] = {
            "id": run.id}
        # print('scanning run history:')
        # history_rows = list(run.scan_history(keys=None))
        # print('creating dataframe:')
        # conf_runs[run.name]["history"] = pd.DataFrame(history_rows)
        conf_runs[run.name]["history"] = run.history(samples=20000)
        conf_runs[run.name]["history"].set_index("_runtime", inplace=True)


conf_runs["entropy_conf"]["display_name"] = "Entropy"
conf_runs["margin_conf"]["display_name"] = "Margin"
conf_runs["energy_conf"]["display_name"] = "Energy"


conf_runs[default_run_name] = {
    "id": default_run_id,
    "history": default_run_history,
    "display_name": "MSP (Default)"
}

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import itertools
import wandb
from matplotlib.lines import Line2D # Import Line2D for custom legend handles
import numpy as np # Import numpy for array manipulation

# Metrics to plot in the single column
metrics_to_plot = [
    {"key": "AGENT/budget", "title": "Budget", "ylim": (-12, 25), "rolling_window_size": 1},
    {"key": "online_inference/known_classif_confidente", "title": "Classification Confidence", "ylim": (-3, 5), "rolling_window_size": 1},
    {"key": "online_inference/zda_classif_confidence", "title": "Anomaly Detection Confidence", "ylim": (-1, 2), "rolling_window_size": 1},
    {"key": "epistemic_actions_per_episode", "title": "Epistemic Actions per Episode", "ylim": (0, 10), "rolling_window_size":1}
]

# Define the two time intervals to plot
time_intervals = [
    {"start_minutes": 5, "end_minutes": 8, "title": "Early Training (5-8 min)"},
    {"start_minutes": 55, "end_minutes": 60, "title": "Mature Training (55-60 min)"}
]

algorithms = [run_dict["display_name"] if "display_name" in run_dict else run_name for run_name, run_dict in conf_runs.items()]
color_map = get_color_cycle(algorithms)

# Create custom legend handles for the entire figure, based on the algorithms
all_legend_handles = []
all_legend_labels = []
for algo in algorithms:
    all_legend_handles.append(Line2D([0], [0], color=color_map[algo], lw=2, alpha=0.8))
    all_legend_labels.append(algo)

# Add the 'optim' horizontal line to the legend if it's relevant for 'epistemic_actions_per_episode'
# Note: This metric is not in the current selection, but kept for robustness if metrics change.
if any(m['key'] == 'epistemic_actions_per_episode' for m in metrics_to_plot):
    all_legend_handles.append(Line2D([0], [0], color='red', linestyle='--', lw=1, alpha=0.5))
    all_legend_labels.append('optim no. of epistemic actions')

# Add a specific legend entry for epistemic actions (stars)
star_handle = Line2D([0], [0], marker='*', color='black', linestyle='None', markersize=15, label='Epistemic Actions') # Increased markersize
all_legend_handles.append(star_handle)
all_legend_labels.append('Epistemic Actions')

# Create the figure and a len(metrics_to_plot) x len(time_intervals) grid of subplots
fig, axes = plt.subplots(len(metrics_to_plot), len(time_intervals), figsize=(20 * len(time_intervals), 5 * len(metrics_to_plot)), sharex=False)

# Ensure axes is always a 2D array for consistent indexing
if len(metrics_to_plot) == 1 and len(time_intervals) == 1:
    axes = np.array([[axes]])
elif len(metrics_to_plot) == 1:
    axes = axes[np.newaxis, :]
elif len(time_intervals) == 1:
    axes = axes[:, np.newaxis]

for col_idx, interval_info in enumerate(time_intervals):
    start_time_secs = interval_info["start_minutes"] * 60
    end_time_secs = interval_info["end_minutes"] * 60
    interval_title = interval_info["title"]

    # --- Episode Reset Calculation (not plotted, but needed to remove axvline) ---
    # This block remains for reference but the plotting part is removed as per user request.
    episode_reset_events_by_run = {algo: [] for algo in algorithms}
    min_separation_seconds = 4

    for run_name, run_dict in conf_runs.items():
        display_name = run_dict.get("display_name", run_name)
        history_df = run_dict["history"].copy()
        history_df = history_df[(history_df.index >= start_time_secs) & (history_df.index <= end_time_secs)]

        if "AGENT/budget" in history_df.columns and not history_df["AGENT/budget"].empty:
            budget_series = history_df["AGENT/budget"].dropna()
            if not budget_series.empty:
                crossed_under_neg10 = (budget_series < -10) & (budget_series.shift(1) >= -10)
                crossed_over_25 = (budget_series > 25) & (budget_series.shift(1) <= 25)
                true_reset_events_mask = crossed_under_neg10 | crossed_over_25
                reset_events = budget_series.index[true_reset_events_mask].tolist()
                filtered_reset_events = []
                if reset_events:
                    sorted_events = sorted(list(set(reset_events)))
                    filtered_reset_events.append(sorted_events[0])
                    for i_event in range(1, len(sorted_events)):
                        if sorted_events[i_event] - filtered_reset_events[-1] > min_separation_seconds:
                            filtered_reset_events.append(sorted_events[i_event])
                episode_reset_events_by_run[display_name].extend(filtered_reset_events);
    # -------------------------------------

    # Loop through each metric in the current column
    for row_idx, metric_info in enumerate(metrics_to_plot):
        metric_key = metric_info["key"]
        metric_title = metric_info["title"]
        metric_ylim = metric_info["ylim"]
        ROLLING_WINDOW_SIZE = metric_info["rolling_window_size"]
        ax = axes[row_idx, col_idx] # Get the current subplot axis based on row and column

        for run_name, run_dict in conf_runs.items():
            display_name = run_dict.get("display_name", run_name)
            history_df = run_dict["history"].copy()

            # Trim the timeseries data to the specified range
            history_df = history_df[(history_df.index >= start_time_secs) & (history_df.index <= end_time_secs)]

            if metric_key in history_df.columns and not history_df[metric_key].empty:
                data_series = history_df[metric_key]

                # Ensure data is numeric before interpolation
                data_series_numeric = pd.to_numeric(data_series, errors='coerce')

                # Interpolate discontinuities and address FutureWarning
                data_series_interpolated = data_series_numeric.infer_objects(copy=False).interpolate(method='linear')

                # Calculate running average
                running_avg_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()

                # Calculate rolling 1st and 3rd quantiles from the interpolated data series
                q1_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.25)
                q3_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.75)

                if metric_key == "AGENT/budget":
                    # Plot raw data (running_avg_series with window=1) with higher transparency
                    ax.plot(running_avg_series.index, running_avg_series,
                             color=color_map[display_name],
                             linewidth=1,
                             alpha=0.5)

                    # Plot a smoother running average with alpha=1.0
                    smooth_budget_window = 100 # New rolling window size for smoother average
                    smooth_avg_series = data_series_interpolated.rolling(window=smooth_budget_window, min_periods=1).mean()
                    ax.plot(smooth_avg_series.index, smooth_avg_series,
                             color=color_map[display_name],
                             linewidth=2,
                             alpha=1.0) # Fully opaque for the main line

                    # Plot 1st and 3rd quantiles with lower alpha
                    ax.plot(q1_series.index, q1_series,
                             color=color_map[display_name],
                             linestyle='--',
                             alpha=0.2,
                             linewidth=0.5)
                    ax.plot(q3_series.index, q3_series,
                             color=color_map[display_name],
                             linestyle='--',
                             alpha=0.2,
                             linewidth=0.5)
                else:
                    # Existing plotting logic for other metrics
                    # Plot Running Average line
                    ax.plot(running_avg_series.index, running_avg_series,
                             color=color_map[display_name],
                             linewidth=2,
                             alpha=0.8)

                    # Plot 1st and 3rd quantiles
                    ax.plot(q1_series.index, q1_series,
                             color=color_map[display_name],
                             linestyle='--',
                             alpha=0.5,
                             linewidth=0.5)
                    ax.plot(q3_series.index, q3_series,
                             color=color_map[display_name],
                             linestyle='--',
                             alpha=0.5,
                             linewidth=0.5)

                # Add stars for Epistemic Actions on Classification Confidence plot
                if metric_key == "online_inference/known_classif_confidente" and "AGENT/Epistemic Actions taken" in history_df.columns \
                  or metric_key == "online_inference/zda_classif_confidence" and "AGENT/Epistemic Actions taken" in history_df.columns:
                    epistemic_actions_series = history_df["AGENT/Epistemic Actions taken"].dropna()
                    if not epistemic_actions_series.empty:
                        # Find points where epistemic actions were taken
                        epistemic_action_times = epistemic_actions_series[epistemic_actions_series > 0].index

                        # Get the corresponding confidence values at these times
                        confidence_at_epistemic_actions = data_series_interpolated.loc[epistemic_action_times]

                        # Scale markersize by the value of 'AGENT/Epistemic Actions taken' for scatter
                        # 's' in scatter is area, so scaling factor needs adjustment from 'markersize' (diameter)
                        scaled_s_sizes = (epistemic_actions_series.loc[epistemic_action_times] * 350) # Scaling factor for area

                        ax.scatter(epistemic_action_times, confidence_at_epistemic_actions,
                                   s=scaled_s_sizes, marker='*', color=color_map[display_name],
                                   alpha=1.0) # Increased alpha

        # Add horizontal line for 'epistemic_actions_per_episode' if it's the current metric
        if metric_key == "epistemic_actions_per_episode": # Assuming this was for a different context or intended for 'mean_episode_budget'
            ax.axhline(y=7, color='red', linestyle='--', label='optim')

        # Removed vertical lines for episode resets as per user request

        # Make axis numbers big and bold
        ax.tick_params(axis='y', labelsize=axis_font_dict['size'])

        # Make tick labels bold
        for tick in ax.get_yticklabels():
            tick.set_fontweight('bold')

        # Control x-axis labels: only for the bottom row
        if row_idx == len(metrics_to_plot) - 1: # Check if it's the very last plot in the column
            ax.tick_params(axis='x', labelsize=axis_font_dict['size'], labelbottom=True) # Ensure labels are shown
            for tick in ax.get_xticklabels():
                tick.set_fontweight('bold')
            ax.set_xlabel('Training Time (seconds)', **axis_font_dict)
        else:
            ax.tick_params(axis='x', labelbottom=False) # Hide x-axis labels for other subplots

        ax.set_title(f"{metric_title} - {interval_title}", **title_font_dict); # Title with metric and interval
        if metric_ylim:
            ax.set_ylim(metric_ylim) # Set y-axis limits if provided
        ax.grid(True)

# Add a single legend to the right of the entire plot
fig.legend(all_legend_handles, all_legend_labels,
           prop={'weight': 'bold', 'size': 20},
           loc='center right', # Position the legend outside the plots
           bbox_to_anchor=(0.29, 0.93), # Adjusted position for single column
           title='Legend',
           title_fontsize=legend_font_dict['size'] + 2,
           ncol=2)

# Add a super title for the entire figure
fig.suptitle('IM Confidence Estimation Strategies Across Training Phases', **title_font_dict, y=1.02)

# Adjust layout to make space for the suptitle and legend
plt.tight_layout(rect=[0, 0, 0.85, 1.03]) # Adjusted right boundary to make space for legend
plt.show()

# Budget Constraints

In [ ]:
budget_runs = {}
for run in runs:
    if run.group == "budget_constraints":
        print(run.name, run.id)
        budget_runs[run.name] = {
            "id": run.id}
        # print('scanning run history:')
        # history_rows = list(run.scan_history(keys=None))
        # print('creating dataframe:')
        # budget_runs[run.name]["history"] = pd.DataFrame(history_rows)
        budget_runs[run.name]["history"] = run.history(samples=20000)
        budget_runs[run.name]["history"].set_index("_runtime", inplace=True)

budget_runs["l-minus20"]["display_name"] = "Min -20 Max +25"
budget_runs["l-minus5_h-35"]["display_name"] = "Min -5 Max +35"
budget_runs["l-minus5_h-40"]["display_name"] = "Min -5 Max +40"
budget_runs["l-minus1"]["display_name"] = "Min -1 Max +25"

budget_runs[default_run_name] = {
    "id": default_run_id,
    "history": default_run_history,
    "display_name": "Min -10 Max +25"
}


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import itertools
import wandb
from matplotlib.lines import Line2D # Import Line2D for custom legend handles

# Metrics from the first plot (Rewards, Budget, Epistemic Actions)
metrics_to_plot_part1 = [
    # {"key": "AGENT/generic_reward", "title": "Rewards", "ylim": (-1.5, 2), "rolling_window_size": 1000},
    {"key": "mean_episode_budget", "title": "Mean Episode Budget", "ylim": (-22, 42), "rolling_window_size": 1000},
    {"key": "epistemic_actions_per_episode", "title": "Epistemic Actions per Episode", "ylim": (0, 12), "rolling_window_size": 1000}
]

# Metrics from the second plot (Inference metrics)
metrics_to_plot_part2 = [
    {"key": "episode_count", "title": "Episodes Done", "ylim": (0, 515), "rolling_window_size": 1},
    {"key": "steps_per_episode", "title": "Steps per episode", "ylim": (0.5, 180), "rolling_window_size": 1000},
    #{"key": "active_inference/value_loss", "title": "Critic Loss", "ylim": (0, 2), "rolling_window_size": 1000}
]



# Combine all metrics for the legend check and general overview (not for direct iteration over axes)
metrics_to_plot = metrics_to_plot_part1 + metrics_to_plot_part2

# max_secs_to_plot from the original code for the first set of plots
max_secs_to_plot = 40 * 60

algorithms = [run_dict["display_name"] if "display_name" in run_dict else run_name for run_name, run_dict in budget_runs.items()]
color_map = get_color_cycle(algorithms)

# Create custom legend handles for the entire figure, based on the algorithms
all_legend_handles = []
all_legend_labels = []
for algo in algorithms:
    all_legend_handles.append(Line2D([0], [0], color=color_map[algo], lw=2, alpha=0.8))
    all_legend_labels.append(algo)

# Add the 'optim' horizontal line to the legend if it's relevant for 'epistemic_actions_per_episode'
if any(m['key'] == 'epistemic_actions_per_episode' for m in metrics_to_plot):
    all_legend_handles.append(Line2D([0], [0], color='red', linestyle='--', lw=1, alpha=0.5))
    all_legend_labels.append('optim no. of epistemic actions')


# Create the figure and a 3x2 grid of subplots
fig, axes = plt.subplots(len(metrics_to_plot_part1), 2, figsize=(20, 8), sharex=False) # Adjust figsize for better readability

# Loop through each column (part1 for col 0, part2 for col 1)
for col_idx, current_metrics_list in enumerate([metrics_to_plot_part1, metrics_to_plot_part2]):
    # Loop through each metric in the current column's list
    for row_idx, metric_info in enumerate(current_metrics_list):
        metric_key = metric_info["key"]
        metric_title = metric_info["title"]
        metric_ylim = metric_info["ylim"]
        ROLLING_WINDOW_SIZE = metric_info["rolling_window_size"]
        ax = axes[row_idx, col_idx] # Get the current subplot axis based on row and column

        for run_name, run_dict in budget_runs.items():
            display_name = run_dict.get("display_name", run_name)
            history_df = run_dict["history"].copy()

            # Trim the timeseries data up to max_secs_to_plot seconds
            history_df = history_df[history_df.index <= max_secs_to_plot]

            if metric_key in history_df.columns and not history_df[metric_key].empty:
                data_series = history_df[metric_key]

                # Ensure data is numeric before interpolation
                data_series_numeric = pd.to_numeric(data_series, errors='coerce')

                # Interpolate discontinuities and address FutureWarning
                data_series_interpolated = data_series_numeric.infer_objects(copy=False).interpolate(method='linear')

                # Calculate running average
                running_avg_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()

                # Calculate rolling 1st and 3rd quantiles from the interpolated data series
                q1_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.25)
                q3_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.75)

                # Plot Running Average line
                ax.plot(running_avg_series.index, running_avg_series,
                         color=color_map[display_name],
                         linewidth=2,
                         alpha=0.8)

                if metric_key != "steps_per_episode" and metric_key !="active_inference/value_loss" \
                  and metric_key != "epistemic_actions_per_episode" and metric_key != "AGENT/generic_reward":

                  # Plot 1st and 3rd quantiles
                  ax.plot(q1_series.index, q1_series,
                          color=color_map[display_name],
                          linestyle='--',
                          alpha=0.9,
                          linewidth=1)
                  ax.plot(q3_series.index, q3_series,
                          color=color_map[display_name],
                          linestyle='--',
                          alpha=0.9,
                          linewidth=1)

        # Add horizontal line for 'epistemic_actions_per_episode' if it's the current metric
        if metric_key == "epistemic_actions_per_episode":
            ax.axhline(y=7, color='red', linestyle='--', label='optim')

        # Make axis numbers big and bold
        ax.tick_params(axis='y', labelsize=axis_font_dict['size'])

        # Make tick labels bold
        for tick in ax.get_yticklabels():
            tick.set_fontweight('bold')

        # Control x-axis labels: only for the bottom row (row_idx == 2)
        if row_idx == 1: # Check if it's the bottom row
            ax.tick_params(axis='x', labelsize=axis_font_dict['size'], labelbottom=True) # Ensure labels are shown
            for tick in ax.get_xticklabels():
                tick.set_fontweight('bold')
            ax.set_xlabel('Training Time (seconds)', **axis_font_dict)
        else:
            ax.tick_params(axis='x', labelbottom=False) # Hide x-axis labels for other subplots

        ax.set_title(metric_title, **title_font_dict)
        # ax.set_ylabel(metric_title, **axis_font_dict)
        if metric_ylim:
            ax.set_ylim(metric_ylim) # Set y-axis limits if provided
        ax.grid(True)

# Add a single legend to the right of the entire plot
fig.legend(all_legend_handles, all_legend_labels,
           prop=legend_font_dict,
           loc='center right', # Position the legend outside the plots
           bbox_to_anchor=(0.75, -0.07), # Example coordinates to place it outside
           title='Budget Settings',
           title_fontsize=legend_font_dict['size'] + 2,
           ncol=3)

# Add a super title for the entire figure
fig.suptitle('Modifying Budget Constraints', **title_font_dict, y=1.02)

# Adjust layout to make space for the suptitle and legend
plt.tight_layout(rect=[0, 0, 1, 1.03]) # Adjust right and top boundary
plt.show()

# Baselines


In [ ]:
baselines_runs = {}
for run in runs:
    if run.group == "agents":
        print(run.name, run.id)
        baselines_runs[run.name] = {
            "id": run.id}
        # print('scanning run history:')
        # history_rows = list(run.scan_history(keys=None))
        # print('creating dataframe:')
        # baselines_runs[run.name]["history"] = pd.DataFrame(history_rows)
        baselines_runs[run.name]["history"] = run.history(samples=20000)
        baselines_runs[run.name]["history"].set_index("_runtime", inplace=True)

baselines_runs["DDQN (Default)"]["display_name"] = "DDQN"
baselines_runs["Greedy-CTI (DDQN)"]["display_name"] = "Greedy"
baselines_runs["Periodic CTI (DDQN)"]["display_name"] = "Periodic"
baselines_runs["DQN"]["display_name"] = "DQN"
baselines_runs["DuelingDQN"]["display_name"] = "DuelingDQN"
baselines_runs["DuelingDDQN"]["display_name"] = "DuelingDDQN"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import itertools
import wandb
from matplotlib.lines import Line2D # Import Line2D for custom legend handles

# Metrics from the first plot (Rewards, Budget, Epistemic Actions)
metrics_to_plot_part1 = [
    {"key": "AGENT/clustering_reward", "title": "Unsupevised Rewards", "ylim": (-3, 0.1), "rolling_window_size": 1200},
    {"key": "AGENT/generic_reward", "title": "Total Rewards", "ylim": (-1, 0.7), "rolling_window_size": 600},


]

# Metrics from the second plot (Inference metrics)
metrics_to_plot_part2 = [
    {"key": "AGENT/classification_reward", "title": "Supervised Rewards", "ylim": (-0.1, 1.83), "rolling_window_size": 600},
     {"key": "mean_episode_budget", "title": "Mean Episode Budget", "ylim": (-10, 25), "rolling_window_size": 600},
    #{"key": "steps_per_episode", "title": "Steps per Episode", "ylim": (10, 250), "rolling_window_size": 600},
    #{"key": "active_inference/value_loss", "title": "Critic Loss", "ylim": (0, 1.5), "rolling_window_size": 600},

]

# Combine all metrics for the legend check and general overview (not for direct iteration over axes)
metrics_to_plot = metrics_to_plot_part1 + metrics_to_plot_part2

# max_secs_to_plot from the original code for the first set of plots
max_secs_to_plot = 60 * 60 # 2700 seconds

algorithms = [run_dict["display_name"] if "display_name" in run_dict else run_name for run_name, run_dict in baselines_runs.items()]
color_map = get_color_cycle(algorithms)

# Create custom legend handles for the entire figure, based on the algorithms
all_legend_handles = []
all_legend_labels = []
for algo in algorithms:
    all_legend_handles.append(Line2D([0], [0], color=color_map[algo], lw=2, alpha=0.8))
    all_legend_labels.append(algo)

# Add the 'optim' horizontal line to the legend if it's relevant for 'epistemic_actions_per_episode'
if any(m['key'] == 'epistemic_actions_per_episode' for m in metrics_to_plot):
    all_legend_handles.append(Line2D([0], [0], color='red', linestyle='--', lw=1, alpha=0.5))
    all_legend_labels.append('optim no. of epistemic actions')


# Create the figure and a 3x2 grid of subplots
fig, axes = plt.subplots(len(metrics_to_plot_part1), 2, figsize=(20, 8), sharex=False) # Adjust figsize for better readability

# Loop through each column (part1 for col 0, part2 for col 1)
for col_idx, current_metrics_list in enumerate([metrics_to_plot_part1, metrics_to_plot_part2]):
    # Loop through each metric in the current column's list
    for row_idx, metric_info in enumerate(current_metrics_list):
        metric_key = metric_info["key"]
        metric_title = metric_info["title"]
        metric_ylim = metric_info["ylim"]
        ROLLING_WINDOW_SIZE = metric_info["rolling_window_size"]
        ax = axes[row_idx, col_idx] # Get the current subplot axis based on row and column

        for run_name, run_dict in baselines_runs.items():
            display_name = run_dict.get("display_name", run_name)
            history_df = run_dict["history"].copy()

            # Trim the timeseries data up to max_secs_to_plot seconds
            history_df = history_df[history_df.index <= max_secs_to_plot]

            if metric_key in history_df.columns and not history_df[metric_key].empty:
                data_series = history_df[metric_key]

                # Ensure data is numeric before interpolation
                data_series_numeric = pd.to_numeric(data_series, errors='coerce')

                # Interpolate discontinuities and address FutureWarning
                data_series_interpolated = data_series_numeric.infer_objects(copy=False).interpolate(method='linear')

                # Calculate running average
                running_avg_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()

                # Calculate rolling 1st and 3rd quantiles from the interpolated data series
                q1_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.25)
                q3_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.75)

                # Plot Running Average line
                ax.plot(running_avg_series.index, running_avg_series,
                         color=color_map[display_name],
                         linewidth=2,
                         alpha=0.75)

                # Plot 1st and 3rd quantiles
                # ax.plot(q1_series.index, q1_series,
                #          color=color_map[display_name],
                #          linestyle='--',
                #          alpha=0.5,
                #          linewidth=0.5)
                # ax.plot(q3_series.index, q3_series,
                #          color=color_map[display_name],
                #          linestyle='--',
                #          alpha=0.5,
                #          linewidth=0.5)

        # Add horizontal line for 'epistemic_actions_per_episode' if it's the current metric
        if metric_key == "epistemic_actions_per_episode":
            ax.axhline(y=7, color='red', linestyle='--', label='optim')

        # Make axis numbers big and bold
        ax.tick_params(axis='y', labelsize=axis_font_dict['size'])

        # Make tick labels bold
        for tick in ax.get_yticklabels():
            tick.set_fontweight('bold')

        # Control x-axis labels: only for the bottom row (row_idx == 2)
        if row_idx == 1: # Check if it's the bottom row
            ax.tick_params(axis='x', labelsize=axis_font_dict['size'], labelbottom=True) # Ensure labels are shown
            for tick in ax.get_xticklabels():
                tick.set_fontweight('bold')
            ax.set_xlabel('Training Time (seconds)', **axis_font_dict)
        else:
            ax.tick_params(axis='x', labelbottom=False) # Hide x-axis labels for other subplots

        ax.set_title(metric_title, **title_font_dict)
        # ax.set_ylabel(metric_title, **axis_font_dict)
        if metric_ylim:
            ax.set_ylim(metric_ylim) # Set y-axis limits if provided
        ax.grid(True)

# Add a single legend to the right of the entire plot
fig.legend(all_legend_handles, all_legend_labels,
           prop={'weight': 'bold', 'size': 18},
           loc='center right', # Position the legend outside the plots
           bbox_to_anchor=(0.99, 0.58), # Example coordinates to place it outside
           title='Algorithms',
           title_fontsize=legend_font_dict['size'] + 4,
           ncol=3)

# Add a super title for the entire figure
fig.suptitle('Training Evolution of DM Agents baselines', **title_font_dict, y=1.02)

# Adjust layout to make space for the suptitle and legend
plt.tight_layout(rect=[0, 0, 1, 1.03]) # Adjust right and top boundary
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import itertools
import wandb
from matplotlib.lines import Line2D # Import Line2D for custom legend handles
import numpy as np # Import numpy for array manipulation

# Metrics to plot in the single column
metrics_to_plot = [
    {"key": "AGENT/budget", "title": "Budget", "ylim": (-12, 30), "rolling_window_size": 1},
    {"key": "mean_episode_reward", "title": "Mean Episode Reward", "ylim": (-1.5, 0.7), "rolling_window_size": 1},
]

# Define the two time intervals to plot
time_intervals = [
    {"start_minutes": 3.7, "end_minutes": 4, "title": "Early Training "},
    {"start_minutes": 41.5, "end_minutes": 42, "title": "Mature Training"}
]

algorithms = [run_dict["display_name"] if "display_name" in run_dict else run_name for run_name, run_dict in baselines_runs.items()]
color_map = get_color_cycle(algorithms)

# Create custom legend handles for the entire figure, based on the algorithms
all_legend_handles = []
all_legend_labels = []
for algo in algorithms:
    all_legend_handles.append(Line2D([0], [0], color=color_map[algo], lw=2, alpha=0.8))
    all_legend_labels.append(algo)

# Add the 'optim' horizontal line to the legend if it's relevant for 'epistemic_actions_per_episode'
# Note: This metric is not in the current selection, but kept for robustness if metrics change.
if any(m['key'] == 'epistemic_actions_per_episode' for m in metrics_to_plot):
    all_legend_handles.append(Line2D([0], [0], color='red', linestyle='--', lw=1, alpha=0.5))
    all_legend_labels.append('optim no. of epistemic actions')

# Add a specific legend entry for epistemic actions (stars)
star_handle = Line2D([0], [0], marker='*', color='black', linestyle='None', markersize=15, label='Epistemic Actions') # Increased markersize
all_legend_handles.append(star_handle)
all_legend_labels.append('Epistemic Actions')

# Create the figure and a len(metrics_to_plot) x len(time_intervals) grid of subplots
fig, axes = plt.subplots(len(metrics_to_plot), len(time_intervals),
                         figsize=(25, 5 * len(metrics_to_plot)), sharex=False, sharey=False,
                         gridspec_kw={'width_ratios': [0.6, 1.4]}) # Adjusted width ratios

# Ensure axes is always a 2D array for consistent indexing
if len(metrics_to_plot) == 1 and len(time_intervals) == 1:
    axes = np.array([[axes]])
elif len(metrics_to_plot) == 1:
    axes = axes[np.newaxis, :]
elif len(time_intervals) == 1:
    axes = axes[:, np.newaxis]

# Define a small margin to include data points before the start_time_secs
margin_secs = 10

for col_idx, interval_info in enumerate(time_intervals):
    start_time_secs = interval_info["start_minutes"] * 60
    end_time_secs = interval_info["end_minutes"] * 60
    interval_title = interval_info["title"]

    # --- Episode Reset Calculation (not plotted, but needed to remove axvline) ---
    # This block remains for reference but the plotting part is removed as per user request.
    episode_reset_events_by_run = {algo: [] for algo in algorithms}
    min_separation_seconds = 4

    for run_name, run_dict in baselines_runs.items():
        display_name = run_dict.get("display_name", run_name)
        history_df = run_dict["history"].copy()
        # Adjust trimming to include a margin before start_time_secs
        history_df = history_df[(history_df.index >= start_time_secs - margin_secs) & (history_df.index <= end_time_secs)]

        if "AGENT/budget" in history_df.columns and not history_df["AGENT/budget"].empty:
            budget_series = history_df["AGENT/budget"].dropna()
            if not budget_series.empty:
                crossed_under_neg10 = (budget_series < -10) & (budget_series.shift(1) >= -10)
                crossed_over_25 = (budget_series > 25) & (budget_series.shift(1) <= 25)
                true_reset_events_mask = crossed_under_neg10 | crossed_over_25
                reset_events = budget_series.index[true_reset_events_mask].tolist()
                filtered_reset_events = []
                if reset_events:
                    sorted_events = sorted(list(set(reset_events)))
                    filtered_reset_events.append(sorted_events[0])
                    for i_event in range(1, len(sorted_events)):
                        if sorted_events[i_event] - filtered_reset_events[-1] > min_separation_seconds:
                            filtered_reset_events.append(sorted_events[i_event])
                episode_reset_events_by_run[display_name].extend(filtered_reset_events);
    # -------------------------------------

    # Loop through each metric in the current column
    for row_idx, metric_info in enumerate(metrics_to_plot):
        metric_key = metric_info["key"]
        metric_title = metric_info["title"]
        metric_ylim = metric_info["ylim"]
        ROLLING_WINDOW_SIZE = metric_info["rolling_window_size"]
        ax = axes[row_idx, col_idx] # Get the current subplot axis based on row and column

        for run_name, run_dict in baselines_runs.items():
            display_name = run_dict.get("display_name", run_name)
            history_df = run_dict["history"].copy()

            # Trim the timeseries data to the specified range, including the margin
            history_df = history_df[(history_df.index >= start_time_secs - margin_secs) & (history_df.index <= end_time_secs)]

            if metric_key in history_df.columns and not history_df[metric_key].empty:
                data_series = history_df[metric_key]

                # Ensure data is numeric before interpolation
                data_series_numeric = pd.to_numeric(data_series, errors='coerce')

                # Interpolate discontinuities and address FutureWarning
                data_series_interpolated = data_series_numeric.infer_objects(copy=False).interpolate(method='linear')

                # Calculate running average
                running_avg_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()

                # Calculate rolling 1st and 3rd quantiles from the interpolated data series
                q1_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.25)
                q3_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.75)

                if metric_key == "AGENT/generic_reward":
                    # Plot raw data (running_avg_series with window=1) with higher transparency
                    # ax.plot(running_avg_series.index, running_avg_series,
                    #          color=color_map[display_name],
                    #          linewidth=1,
                    #          alpha=0.5)

                    # Plot a smoother running average with alpha=1.0
                    smooth_budget_window = 100 # New rolling window size for smoother average
                    smooth_avg_series = data_series_interpolated.rolling(window=smooth_budget_window, min_periods=1).mean()
                    ax.plot(smooth_avg_series.index, smooth_avg_series,
                             color=color_map[display_name],
                             linewidth=2,
                             alpha=1.0) # Fully opaque for the main line

                    # Plot 1st and 3rd quantiles with lower alpha
                    # ax.plot(q1_series.index, q1_series,
                    #          color=color_map[display_name],
                    #          linestyle='--',
                    #          alpha=0.2,
                    #          linewidth=0.5)
                    # ax.plot(q3_series.index, q3_series,
                    #          color=color_map[display_name],
                    #          linestyle='--',
                    #          alpha=0.2,
                    #          linewidth=0.5)
                else:
                    # Existing plotting logic for other metrics
                    # Plot Running Average line
                    ax.plot(running_avg_series.index, running_avg_series,
                             color=color_map[display_name],
                             linewidth=2,
                             alpha=0.8)

                    # Plot 1st and 3rd quantiles
                    ax.plot(q1_series.index, q1_series,
                             color=color_map[display_name],
                             linestyle='--',
                             alpha=0.5,
                             linewidth=0.5)
                    ax.plot(q3_series.index, q3_series,
                             color=color_map[display_name],
                             linestyle='--',
                             alpha=0.5,
                             linewidth=0.5)


                # Add stars for Epistemic Actions on budget plot
                if metric_key == "AGENT/budget" and "AGENT/Epistemic Actions taken" in history_df.columns:
                    epistemic_actions_series = history_df["AGENT/Epistemic Actions taken"].dropna()
                    if not epistemic_actions_series.empty:
                        # Find points where epistemic actions were taken
                        epistemic_action_times = epistemic_actions_series[epistemic_actions_series > 0].index

                        # Get the corresponding values at these times
                        values_at_epistemic_actions = running_avg_series.loc[epistemic_action_times]

                        # Scale markersize by the value of 'AGENT/Epistemic Actions taken' for scatter
                        # 's' in scatter is area, so scaling factor needs adjustment from 'markersize' (diameter)
                        scaled_s_sizes = (epistemic_actions_series.loc[epistemic_action_times] * 350) # Scaling factor for area

                        ax.scatter(epistemic_action_times, values_at_epistemic_actions,
                                   s=scaled_s_sizes, marker='*', color=color_map[display_name],
                                   alpha=1.0) # Increased alpha

        # Add horizontal line for 'epistemic_actions_per_episode' if it's the current metric
        if metric_key == "epistemic_actions_per_episode": # Assuming this was for a different context or intended for 'mean_episode_budget'
            ax.axhline(y=7, color='red', linestyle='--', label='optim')

        # Removed vertical lines for episode resets as per user request

        # Make axis numbers big and bold
        ax.tick_params(axis='y', labelsize=axis_font_dict['size'])

        # Make tick labels bold
        for tick in ax.get_yticklabels():
            tick.set_fontweight('bold')

        # Control x-axis labels: only for the bottom row
        if row_idx == len(metrics_to_plot) - 1: # Check if it's the very last plot in the column
            ax.tick_params(axis='x', labelsize=axis_font_dict['size'], labelbottom=True) # Ensure labels are shown
            for tick in ax.get_xticklabels():
                tick.set_fontweight('bold')
            ax.set_xlabel('Training Time (seconds)', **axis_font_dict)
        else:
            ax.tick_params(axis='x', labelbottom=False) # Hide x-axis labels for other subplots

        # Set x-axis limits to explicitly start from start_time_secs
        ax.set_xlim(start_time_secs, end_time_secs)


        # Set title to include metric and interval, but without fixed ylim.
        ax.set_title(f"{metric_title} - {interval_title}", {'weight': 'bold', 'size': 20}); # Title with metric and interval
        if metric_ylim:
            ax.set_ylim(metric_ylim) # Allow y-axis to be dynamically determine
            if metric_key == "mean_episode_reward" and  col_idx == 1:
              ax.set_ylim(0,0.7)
        ax.grid(True)

# Add a single legend to the right of the entire plot
fig.legend(all_legend_handles, all_legend_labels,
           prop={'weight': 'bold', 'size': 20},
           loc='center right', # Position the legend outside the plots
           bbox_to_anchor=(0.85, 0.2), # Adjusted position for single column
           title='Algorithms', # Changed title to 'Algorithms'
           title_fontsize=legend_font_dict['size'] + 2,
           ncol=2)

# Add a super title for the entire figure
fig.suptitle('Baselines Performance across Training Phases', **title_font_dict, y=1.02)

# Adjust layout to make space for the suptitle and legend
plt.tight_layout(rect=[0, 0, 0.85, 1.03]) # Adjusted right boundary to make space for legend
plt.show()

# The effect of low high CTI price on performance

In [ ]:
cti_price_runs = {}
for run in runs:
    if run.group == "cti_prices_bis":
        print(run.name, run.id)
        cti_price_runs[run.name] = {
            "id": run.id}
        # print('scanning run history:')
        # history_rows = list(run.scan_history(keys=None))
        # print('creating dataframe:')
        # cti_price_runs[run.name]["history"] = pd.DataFrame(history_rows)
        cti_price_runs[run.name]["history"] = run.history(samples=20000)
        cti_price_runs[run.name]["history"].set_index("_runtime", inplace=True)


cti_price_runs["cti_price_factor_100"]["display_name"] = "x100"
cti_price_runs["cti_price_factor_75"]["display_name"] = "x75"
cti_price_runs["cti_price_factor_200"]["display_name"] = "x200"

cti_price_runs[default_run_name] = {
    "id": default_run_id,
    "history": default_run_history,
    "display_name": "x50"
}

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import itertools
import wandb
from matplotlib.lines import Line2D # Import Line2D for custom legend handles

# Metrics from the first plot (Rewards, Budget, Epistemic Actions)
metrics_to_plot_part1 = [
    {"key": "AGENT/generic_reward", "title": "Rewards", "ylim": (-1, 0.7), "rolling_window_size": 600},
    {"key": "mean_episode_budget", "title": "Mean Episode Budget", "ylim": (-12, 28), "rolling_window_size": 600},
]

# Metrics from the second plot (Inference metrics)
metrics_to_plot_part2 = [
    {"key": "epistemic_actions_per_episode", "title": "Epistemic Actions per Episode", "ylim": (3, 12), "rolling_window_size": 600},
    {"key": "AGENT/known traffic action", "title": "Obey to IM classification", "ylim": (0.4, 1.05), "rolling_window_size":600},
]

# Combine all metrics for the legend check and general overview (not for direct iteration over axes)
metrics_to_plot = metrics_to_plot_part1 + metrics_to_plot_part2

# max_secs_to_plot from the original code for the first set of plots
max_secs_to_plot = 60 * 60 # 2700 seconds

algorithms = [run_dict["display_name"] if "display_name" in run_dict else run_name for run_name, run_dict in cti_price_runs.items()]
color_map = get_color_cycle(algorithms)

# Create custom legend handles for the entire figure, based on the algorithms
all_legend_handles = []
all_legend_labels = []
for algo in algorithms:
    all_legend_handles.append(Line2D([0], [0], color=color_map[algo], lw=2, alpha=0.8))
    all_legend_labels.append(algo)

# Add the 'optim' horizontal line to the legend if it's relevant for 'epistemic_actions_per_episode'
if any(m['key'] == 'epistemic_actions_per_episode' for m in metrics_to_plot):
    all_legend_handles.append(Line2D([0], [0], color='red', linestyle='--', lw=1, alpha=0.5))
    all_legend_labels.append('optim no. of epistemic actions')


# Create the figure and a 3x2 grid of subplots
fig, axes = plt.subplots(len(metrics_to_plot_part1), 2, figsize=(20, 8), sharex=False) # Adjust figsize for better readability

# Loop through each column (part1 for col 0, part2 for col 1)
for col_idx, current_metrics_list in enumerate([metrics_to_plot_part1, metrics_to_plot_part2]):
    # Loop through each metric in the current column's list
    for row_idx, metric_info in enumerate(current_metrics_list):
        metric_key = metric_info["key"]
        metric_title = metric_info["title"]
        metric_ylim = metric_info["ylim"]
        ROLLING_WINDOW_SIZE = metric_info["rolling_window_size"]
        ax = axes[row_idx, col_idx] # Get the current subplot axis based on row and column

        for run_name, run_dict in cti_price_runs.items():
            display_name = run_dict.get("display_name", run_name)
            history_df = run_dict["history"].copy()

            # Trim the timeseries data up to max_secs_to_plot seconds
            history_df = history_df[history_df.index <= max_secs_to_plot]

            if metric_key in history_df.columns and not history_df[metric_key].empty:
                data_series = history_df[metric_key]

                # Ensure data is numeric before interpolation
                data_series_numeric = pd.to_numeric(data_series, errors='coerce')

                # Interpolate discontinuities and address FutureWarning
                data_series_interpolated = data_series_numeric.infer_objects(copy=False).interpolate(method='linear')

                # Calculate running average
                running_avg_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()

                # Calculate rolling 1st and 3rd quantiles from the interpolated data series
                q1_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.25)
                q3_series = data_series_interpolated.rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).quantile(0.75)

                # Plot Running Average line
                if metric_key == "AGENT/known traffic action":
                  running_avg_series = (2 - running_avg_series) / 2
                  q1_series
                ax.plot(running_avg_series.index, running_avg_series,
                         color=color_map[display_name],
                         linewidth=2,
                         alpha=0.8)

                # # Plot 1st and 3rd quantiles
                # ax.plot(q1_series.index, q1_series,
                #          color=color_map[display_name],
                #          linestyle='--',
                #          alpha=0.5,
                #          linewidth=0.5)
                # ax.plot(q3_series.index, q3_series,
                #          color=color_map[display_name],
                #          linestyle='--',
                #          alpha=0.5,
                #          linewidth=0.5)

        # Add horizontal line for 'epistemic_actions_per_episode' if it's the current metric
        if metric_key == "epistemic_actions_per_episode":
            ax.axhline(y=7, color='red', linestyle='--', label='optim')

        # Make axis numbers big and bold
        ax.tick_params(axis='y', labelsize=axis_font_dict['size'])

        # Make tick labels bold
        for tick in ax.get_yticklabels():
            tick.set_fontweight('bold')

        # Control x-axis labels: only for the bottom row (row_idx == 2)
        if row_idx == 2: # Check if it's the bottom row
            ax.tick_params(axis='x', labelsize=axis_font_dict['size'], labelbottom=True) # Ensure labels are shown
            for tick in ax.get_xticklabels():
                tick.set_fontweight('bold')
            ax.set_xlabel('Training Time (seconds)', **axis_font_dict)
        else:
            ax.tick_params(axis='x', labelbottom=False) # Hide x-axis labels for other subplots

        ax.set_title(metric_title, **title_font_dict)
        # ax.set_ylabel(metric_title, **axis_font_dict)
        if metric_ylim:
            ax.set_ylim(metric_ylim) # Set y-axis limits if provided
        ax.grid(True)

# Add a single legend to the right of the entire plot
fig.legend(all_legend_handles, all_legend_labels,
           prop=legend_font_dict,
           loc='center right', # Position the legend outside the plots
           bbox_to_anchor=(0.95, 0.1), # Example coordinates to place it outside
           title='CTI price factor',
           title_fontsize=legend_font_dict['size'] + 2,
           ncol=3)

# Add a super title for the entire figure
fig.suptitle('Increasing the scale of CTI prices', **title_font_dict, y=1.02)

# Adjust layout to make space for the suptitle and legend
plt.tight_layout(rect=[0, 0, 1, 1.03]) # Adjust right and top boundary
plt.show()

# Sensitivity Analisis


In [ ]:
rainbow_runs = {}
for run in runs:
    if run.group == "rainbow":
        print(run.name, run.id)
        rainbow_runs[run.name] = {
            "id": run.id}
        # print('scanning run history:')
        # history_rows = list(run.scan_history(keys=None))
        # print('creating dataframe:')
        # rainbow_runs[run.name]["history"] = pd.DataFrame(history_rows)
        rainbow_runs[run.name]["history"] = run.history(samples=20000)
        rainbow_runs[run.name]["history"].set_index("_runtime", inplace=True)

rainbow_runs['no_soft_update']['display_name'] = 'PER'
rainbow_runs['no_per']['display_name'] = 'Soft Update'
rainbow_runs['no_per_no_soft_update']['display_name'] = 'Baseline'
rainbow_runs[default_run_name] = {
    "id": default_run_id,
    "history": default_run_history,
    "display_name": "PER + Soft Update"
}

In [ ]:
statistics_data = []
metrics_to_analyze = [
    "mean_episode_reward",
    "steps_per_episode",
    "sum_episode_rewards",
    "mean_episode_budget"
]
mature_training_start_seconds = 30 * 60 # 30 minutes

for run_name, run_dict in rainbow_runs.items(): # Corrected to use rainbow_runs
    display_name = run_dict.get("display_name", run_name)
    if "history" not in run_dict:
        print(f"Skipping run {run_name} because 'history' data is missing.")
        continue
    history_df = run_dict["history"].copy()

    # Filter for mature training (after 30 minutes)
    mature_history_df = history_df[history_df.index >= mature_training_start_seconds]

    for metric_key in metrics_to_analyze:
        if metric_key in mature_history_df.columns and not mature_history_df[metric_key].empty:
            series = mature_history_df[metric_key].dropna()
            if not series.empty:
                statistics_data.append({
                    "Algorithm": display_name,
                    "Metric": metric_key,
                    "Mean": series.mean(),
                    "Std Dev": series.std(),
                    "Min": series.min(),
                    "25th Pctl": series.quantile(0.25),
                    "Median": series.median(),
                    "75th Pctl": series.quantile(0.75),
                    "Max": series.max()
                })

if statistics_data:
    stats_df = pd.DataFrame(statistics_data)
    # Optional: Format for better display
    stats_df_styled = stats_df.style.format({
        "Mean": "{:.3f}",
        "Std Dev": "{:.3f}",
        "Min": "{:.3f}",
        "25th Pctl": "{:.3f}",
        "Median": "{:.3f}",
        "75th Pctl": "{:.3f}",
        "Max": "{:.3f}"
    })
    display(stats_df_styled)
else:
    print("No data found for the specified metrics in rainbow_runs after 30 minutes.")


In [ ]:
update_target_freq_runs = {}
for run in runs:
    if run.group == "update_target_freq":
        print(run.name, run.id)
        update_target_freq_runs[run.name] = {
            "id": run.id}
        # print('scanning run history:')
        # history_rows = list(run.scan_history(keys=None))
        # print('creating dataframe:')
        # update_target_freq_runs[run.name]["history"] = pd.DataFrame(history_rows)
        update_target_freq_runs[run.name]["history"] = run.history(samples=20000)
        update_target_freq_runs[run.name]["history"].set_index("_runtime", inplace=True)

update_target_freq_runs['update_target_freq30']['display_name'] = 'Update Target Freq 30'
update_target_freq_runs['update_target_freq70']['display_name'] = 'Update Target Freq 70'
update_target_freq_runs['update_target_freq100']['display_name'] = 'Update Target Freq 100'
update_target_freq_runs[default_run_name] = {
    "id": default_run_id,
    "history": default_run_history,
    "display_name": "Update Target Freq 50"
}


In [ ]:
statistics_data = []
metrics_to_analyze = [
    "mean_episode_reward",
    "steps_per_episode",
    "sum_episode_rewards",
    "mean_episode_budget"
]
mature_training_start_seconds = 30 * 60 # 30 minutes

for run_name, run_dict in update_target_freq_runs.items(): # Corrected to use update_target_freq_runs
    display_name = run_dict.get("display_name", run_name)
    if "history" not in run_dict:
        print(f"Skipping run {run_name} because 'history' data is missing.")
        continue
    history_df = run_dict["history"].copy()

    # Filter for mature training (after 30 minutes)
    mature_history_df = history_df[history_df.index >= mature_training_start_seconds]

    for metric_key in metrics_to_analyze:
        if metric_key in mature_history_df.columns and not mature_history_df[metric_key].empty:
            # Convert to numeric and then drop NaN values to handle mixed types
            series = pd.to_numeric(mature_history_df[metric_key], errors='coerce').dropna()
            if not series.empty:
                statistics_data.append({
                    "Algorithm": display_name,
                    "Metric": metric_key,
                    "Mean": series.mean(),
                    "Std Dev": series.std(),
                    "Min": series.min(),
                    "25th Pctl": series.quantile(0.25),
                    "Median": series.median(),
                    "75th Pctl": series.quantile(0.75),
                    "Max": series.max()
                })

if statistics_data:
    stats_df = pd.DataFrame(statistics_data)
    # Optional: Format for better display
    stats_df_styled = stats_df.style.format({
        "Mean": "{:.3f}",
        "Std Dev": "{:.3f}",
        "Min": "{:.3f}",
        "25th Pctl": "{:.3f}",
        "Median": "{:.3f}",
        "75th Pctl": "{:.3f}",
        "Max": "{:.3f}"
    })
    display(stats_df_styled)
else:
    print("No data found for the specified metrics in rainbow_runs after 30 minutes.")


In [ ]:
nstep_rewards_runs = {}
for run in runs:
    if run.group == "nstep_rewards":
        print(run.name, run.id)
        nstep_rewards_runs[run.name] = {
            "id": run.id}
        # print('scanning run history:')
        # history_rows = list(run.scan_history(keys=None))
        # print('creating dataframe:')
        # nstep_rewards_runs[run.name]["history"] = pd.DataFrame(history_rows)
        nstep_rewards_runs[run.name]["history"] = run.history(samples=20000)
        nstep_rewards_runs[run.name]["history"].set_index("_runtime", inplace=True)


